# 00 — Encode chips to embeddings (S1 → CROMA, S2 → TerraMind)

Produces the inputs for `09_embed_DiD.ipynb` (the collaborator's equal-weight DiD with
the representation swapped): each site × period × sensor becomes ONE 768-dimensional
embedding, stored as a **768-band, 1×1-pixel GeoTIFF** in a tree mirroring
`REAP/data/finals/`, so the feature-based notebook's `find_image`/`read_raster` work unchanged.

Approved contract (Jun, 2026-08-14):

| | S1 → CROMA-base (`modality='SAR'`) | S2 → TerraMind-v1-base (`S2L2A` subset) |
|---|---|---|
| bands fed | VV, VH (drop derived VV−VH) | B2,B3,B4,B8,B11,B12 → BLUE, GREEN, RED, NIR_BROAD, SWIR_1, SWIR_2 (drop NDVI/NDWI) |
| nodata fill | per-band chip mean, before padding | per-band chip mean, before resize |
| geometry | reflect-pad 101→104 (native 10 m) | bilinear resize 101→224 (fixed-224 model) |
| value scaling | CROMA recipe: rescale to [0,1] via mean±2σ per channel, constants from the 260-control pool, frozen below | ×10,000 to DN scale, then standardize with TerraMind's own v1 per-band mean/std (the ViT backbone does NOT auto-standardize) |
| output | `SAR_GAP` pooled vector (768) | mean-pooled last-layer tokens (768) |

No fusion: each site gets two separate 768-d vectors (S2 arm, S1 arm), mirroring the feature-based
per-sensor pipeline. All constants + versions land in `encoding_manifest.json`.
Weights: `antofuller/CROMA` (MIT) and `ibm-esa-geospatial/TerraMind-1.0-base`
(Apache-2.0) via the HF cache.

**Storage format note.** A raster's two roles here:
- The feature-based data: few bands, many pixels — shape (8, 101, 101). The spatial grid carries the information.
- Our embeddings: many bands, one pixel — shape (768, 1, 1). Each "band" is one coordinate of
  the embedding vector; the single "pixel" has no spatial meaning. This is what lets the feature-based
  band-count-agnostic code consume embeddings unchanged.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_bounds
import tifffile
import torch
import torch.nn.functional as F

ROOT = Path("/data/wang/junh/githubs/latent-synthetic-control")
DATA = ROOT / "REAP" / "data"
FINALS = DATA / "finals"
EMB = DATA / "embeddings"
EMB_FINALS = EMB / "finals"
HERE = ROOT / "REAP" / "notebooks" / "embed_DiD"

D = 768                      # embedding dimension, both encoders
S2_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDWI"]
S1_BANDS = ["VV", "VH", "VV_minus_VH"]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

# site list — ALWAYS from the matching table (104 stray unranked TIFFs exist on disk)
mt = pd.read_csv(FINALS / "site_matching_table.csv")
treatments = sorted(mt["treatment_site_id"].astype(str).unique())
controls = sorted(mt["counterfactual_site_id"].astype(str).unique())
sites = treatments + controls
print(f"{len(treatments)} treatments + {len(controls)} controls = {len(sites)} sites")


def chip_path(site_id, sensor, period):
    group = "treatment" if site_id.startswith("treatment") else "counterfactual"
    return FINALS / sensor / group / period / f"{site_id}_{period}_{sensor}.tif"


def read_chip(path):
    """(H, W, C) float32; the chips encode nodata as -Infinity -> NaN."""
    arr = tifffile.imread(path).astype(np.float32)
    arr[~np.isfinite(arr)] = np.nan
    return arr

device: cuda:0
26 treatments + 260 controls = 286 sites


## S1 normalization constants — CROMA recipe, frozen from the control pool

CROMA's published preprocessing (`README.md`, "taken from SatMAE and SeCo"): per channel,
`min = mean − 2σ`, `max = mean + 2σ`, rescale to [0,1], clip. Their example computes the
statistics over the batch; we compute them ONCE over all 260 control chips × both periods
(valid pixels only) and apply the same constants to every chip — treatment sites never
influence the normalization.

In [2]:
s1_stats_file = EMB / "s1_norm_stats.json"
if s1_stats_file.exists():
    s1_stats = json.loads(s1_stats_file.read_text())
else:
    acc = {b: [] for b in ("VV", "VH")}
    for sid in controls:
        for period in ("before", "after"):
            chip = read_chip(chip_path(sid, "sentinel1", period))
            for bi, b in enumerate(("VV", "VH")):
                v = chip[..., bi]
                acc[b].append(v[np.isfinite(v)])
    s1_stats = {}
    for b in ("VV", "VH"):
        allv = np.concatenate(acc[b])
        s1_stats[b] = {"mean": float(allv.mean()), "std": float(allv.std()),
                       "n_pixels": int(allv.size)}
    EMB.mkdir(parents=True, exist_ok=True)
    s1_stats_file.write_text(json.dumps(s1_stats, indent=2))
print(json.dumps(s1_stats, indent=2))

{
  "VV": {
    "mean": -9.967060089111328,
    "std": 2.9379823207855225,
    "n_pixels": 5297400
  },
  "VH": {
    "mean": -16.063688278198242,
    "std": 2.981891632080078,
    "n_pixels": 5297400
  }
}


## S2 constants — TerraMind's own v1 pretraining mean/std

Imported from terratorch's source (no transcription), then subset to our six bands.
Our chips store L2A reflectance as 0–1 floats; TerraMind's constants are in DN scale
(reflectance × 10,000), so chips are multiplied by 10,000 first.

In [3]:
from terratorch.models.backbones.terramind.model.terramind_register import (
    PRETRAINED_BANDS, v1_pretraining_mean, v1_pretraining_std)

TM_KEY = "untok_sen2l2a@224"
TM_ALL = PRETRAINED_BANDS[TM_KEY]
BANDS6 = ["BLUE", "GREEN", "RED", "NIR_BROAD", "SWIR_1", "SWIR_2"]  # = B2,B3,B4,B8,B11,B12
IDX6 = [TM_ALL.index(b) for b in BANDS6]
TM_MEAN = np.array([v1_pretraining_mean[TM_KEY][i] for i in IDX6], dtype=np.float32)
TM_STD = np.array([v1_pretraining_std[TM_KEY][i] for i in IDX6], dtype=np.float32)
print("band -> index / mean / std")
for b, i, m, s in zip(BANDS6, IDX6, TM_MEAN, TM_STD):
    print(f"  {b:10s} {i:2d} {m:9.2f} {s:9.2f}")

band -> index / mean / std
  BLUE        1   1503.32   2141.11
  GREEN       2   1718.20   2038.97
  RED         3   1853.91   2134.14
  NIR_BROAD   7   3083.23   1871.92
  SWIR_1     10   2424.88   1434.26
  SWIR_2     11   1857.65   1334.31


## Load both encoders (frozen, eval mode)

In [4]:
sys.path.insert(0, str(HERE))
from use_croma import PretrainedCROMA
from huggingface_hub import hf_hub_download
from terratorch.registry import BACKBONE_REGISTRY

croma_w = hf_hub_download(repo_id="antofuller/CROMA", filename="CROMA_base.pt")
croma = PretrainedCROMA(pretrained_path=croma_w, size="base", modality="SAR",
                        image_resolution=104).to(device).eval()

terramind = BACKBONE_REGISTRY.build(
    "terramind_v1_base", pretrained=True, modalities=["S2L2A"],
    bands={"S2L2A": BANDS6},
).to(device).eval()
for p in croma.parameters():
    p.requires_grad_(False)
for p in terramind.parameters():
    p.requires_grad_(False)
print("encoders loaded")

Initializing SAR encoder


encoders loaded


## Preprocessing + encoding

`preprocess_s1`: chip (101,101,3) → tensor (2,104,104): keep VV,VH; NaN → per-band chip
mean; CROMA [0,1] rescale with the frozen pool constants; reflect-pad 101→104.
`preprocess_s2`: chip (101,101,8) → tensor (6,224,224): keep the 6 measured bands;
NaN → per-band chip mean; ×10,000; standardize with TerraMind v1 constants; bilinear
resize 101→224.

In [5]:
def fill_nan_with_band_mean(x):
    """x: (C, H, W) float32 numpy; NaN -> per-band chip mean (0 if band all-NaN)."""
    for c in range(x.shape[0]):
        band = x[c]
        m = np.isnan(band)
        if m.any():
            fill = np.nanmean(band) if not np.isnan(band).all() else 0.0
            band[m] = fill
    return x


def preprocess_s1(chip):
    x = np.ascontiguousarray(chip[..., :2].transpose(2, 0, 1))  # (2,101,101) VV,VH
    x = fill_nan_with_band_mean(x)
    for c, b in enumerate(("VV", "VH")):
        mean, std = s1_stats[b]["mean"], s1_stats[b]["std"]
        lo, hi = mean - 2 * std, mean + 2 * std
        x[c] = np.clip((x[c] - lo) / (hi - lo), 0.0, 1.0)
    t = torch.from_numpy(x)
    return F.pad(t.unsqueeze(0), (1, 2, 1, 2), mode="reflect").squeeze(0)  # (2,104,104)


def preprocess_s2(chip):
    x = np.ascontiguousarray(chip[..., :6].transpose(2, 0, 1))  # (6,101,101)
    x = fill_nan_with_band_mean(x)
    x = x * 10_000.0
    x = (x - TM_MEAN[:, None, None]) / TM_STD[:, None, None]
    t = torch.from_numpy(x).unsqueeze(0)
    t = F.interpolate(t, size=(224, 224), mode="bilinear", align_corners=False)
    return t.squeeze(0)  # (6,224,224)


@torch.no_grad()
def encode_batch(sensor, tensors):
    batch = torch.stack(tensors).to(device)
    if sensor == "sentinel1":
        return croma(SAR_images=batch)["SAR_GAP"].cpu().numpy()
    out = terramind({"S2L2A": batch})
    return out[-1].mean(dim=1).cpu().numpy()  # mean-pool last-layer tokens

## Determinism check
Encode the same chip twice (separate forward passes) — must be bit-identical.

In [6]:
chk = read_chip(chip_path(treatments[0], "sentinel1", "before"))
a = encode_batch("sentinel1", [preprocess_s1(chk.copy())])
b = encode_batch("sentinel1", [preprocess_s1(chk.copy())])
chk2 = read_chip(chip_path(treatments[0], "sentinel2", "before"))
c = encode_batch("sentinel2", [preprocess_s2(chk2.copy())])
d = encode_batch("sentinel2", [preprocess_s2(chk2.copy())])
print("S1 repeat max|diff|:", float(np.abs(a - b).max()),
      "| S2 repeat max|diff|:", float(np.abs(c - d).max()))
assert np.array_equal(a, b) and np.array_equal(c, d)
print("deterministic; S1 dim", a.shape, "S2 dim", c.shape)

S1 repeat max|diff|: 0.0 | S2 repeat max|diff|: 0.0
deterministic; S1 dim (1, 768) S2 dim (1, 768)


## Encode everything and write the embedding GeoTIFF tree

Each output file: 768 bands × 1×1 pixel, float32, CRS and full-chip bounds carried over
from the source chip (collapsed to a single pixel), nodata NaN — same filenames as
`finals/`, so the feature-based notebook 09 needs only its `BASE_DIR` changed.

In [7]:
# Mirror the matching table into the embeddings tree: the feature-based notebook 09 locates it
# at FINALS_DIR / "site_matching_table.csv", and with BASE_DIR -> embeddings the
# tree must carry the same file (data mirroring instead of a 5th code edit).
import shutil
EMB_FINALS.mkdir(parents=True, exist_ok=True)
shutil.copy2(FINALS / "site_matching_table.csv", EMB_FINALS / "site_matching_table.csv")
print("mirrored site_matching_table.csv")

PREP = {"sentinel1": preprocess_s1, "sentinel2": preprocess_s2}
BATCH = 128
n_written = 0
fill_counts = []

for sensor in ("sentinel1", "sentinel2"):
    for period in ("before", "after"):
        todo, metas = [], []
        for sid in sites:
            src_p = chip_path(sid, sensor, period)
            group = "treatment" if sid.startswith("treatment") else "counterfactual"
            out_p = (EMB_FINALS / sensor / group / period /
                     f"{sid}_{period}_{sensor}.tif")
            todo.append(src_p)
            metas.append((sid, group, out_p))
        vecs = np.zeros((len(todo), D), dtype=np.float32)
        for start in range(0, len(todo), BATCH):
            chunk = todo[start:start + BATCH]
            tensors = []
            for p in chunk:
                chip = read_chip(p)
                nan_ct = int(np.isnan(chip[..., 0]).sum())
                fill_counts.append({"file": p.name, "sensor": sensor,
                                    "period": period, "nodata_pixels": nan_ct})
                tensors.append(PREP[sensor](chip))
            vecs[start:start + len(chunk)] = encode_batch(sensor, tensors)
        assert np.isfinite(vecs).all(), f"non-finite embeddings in {sensor}/{period}"
        for (sid, group, out_p), vec in zip(metas, vecs):
            out_p.parent.mkdir(parents=True, exist_ok=True)
            with rasterio.open(chip_path(sid, sensor, period)) as src:
                crs, bounds = src.crs, src.bounds
            profile = {"driver": "GTiff", "height": 1, "width": 1, "count": D,
                       "dtype": "float32", "crs": crs, "nodata": np.nan,
                       "transform": from_bounds(*bounds, 1, 1), "compress": "lzw"}
            with rasterio.open(out_p, "w", **profile) as dst:
                dst.write(vec.reshape(D, 1, 1))
            n_written += 1
        print(f"{sensor}/{period}: encoded {len(todo)} sites")
print("total embedding files written:", n_written)

mirrored site_matching_table.csv


sentinel1/before: encoded 286 sites


sentinel1/after: encoded 286 sites


sentinel2/before: encoded 286 sites


sentinel2/after: encoded 286 sites
total embedding files written: 1144


## Flat caches + manifest (for later SC / analysis use)

In [8]:
rows = []
for sensor in ("sentinel1", "sentinel2"):
    for period in ("before", "after"):
        for sid in sites:
            group = "treatment" if sid.startswith("treatment") else "counterfactual"
            p = EMB_FINALS / sensor / group / period / f"{sid}_{period}_{sensor}.tif"
            with rasterio.open(p) as src:  # rasterio: tifffile lacks imagecodecs for LZW
                vec = src.read().reshape(-1)
            rows.append({"site_id": sid, "sensor": sensor, "period": period,
                         **{f"emb_{i:03d}": float(v) for i, v in enumerate(vec)}})
cache = pd.DataFrame(rows)
cache.to_csv(EMB / "site_embeddings.csv", index=False)
print("cache:", cache.shape)

import terratorch, rasterio as rio
from importlib.metadata import version
manifest = {
    "date": "2026-08-14",
    "embedding_dim": D,
    "s1_encoder": {"model": "CROMA-base modality=SAR", "weights": str(croma_w),
                   "input": "VV,VH dB; NaN->band mean; [0,1] via mean±2σ (control pool); reflect-pad 101->104",
                   "pooling": "SAR_GAP", "norm_stats": s1_stats},
    "s2_encoder": {"model": "TerraMind-1.0-base S2L2A subset", "bands": BANDS6,
                   "input": "reflectance x10000; TerraMind v1 per-band standardization (manual); bilinear 101->224",
                   "pooling": "mean over last-layer tokens",
                   "tm_mean": TM_MEAN.tolist(), "tm_std": TM_STD.tolist()},
    "versions": {"torch": torch.__version__, "terratorch": version("terratorch"),
                 "rasterio": rio.__version__, "numpy": np.__version__},
    "n_files": n_written,
}
(EMB / "encoding_manifest.json").write_text(json.dumps(manifest, indent=2))
pd.DataFrame(fill_counts).to_csv(EMB / "nodata_fill_counts.csv", index=False)
nz = [f for f in fill_counts if f["nodata_pixels"] > 0]
print(f"manifest written; chips with any nodata filled: {len(nz)} / {len(fill_counts)}")

cache: (1144, 771)
manifest written; chips with any nodata filled: 242 / 1144
